In [ ]:
import traceback
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import pandas as pd
import re
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

driver = webdriver.Edge()
driver.get("https://www.google.com/search?client=firefox-b-d&hs=xsi&sca_esv=2a9cf1d07f23bfb1&udm=8&fbs=ADc_l-aN0CWEZBOHjofHoaMMDiKpyjlKjHcf0Gfll8GFgal4hehfZi4UFGoO13qzQIwcqjzu2rIstdd54-Dcz0oF6WpXFYNhE0lv41bxflkcM5ABWLuPfz9RSVOcpFITOHH-aoVw92hdQiYTTBo4IMxyHf18nV1NOFl5NYLCOkbo41ouVGewEX1gy5FLocB3poBsaLLx-ypzNmyro0gGGlij5Ht3tamx_Q&q=data+scientist+jobs&sa=X&ved=2ahUKEwiQ2Yar5_6UAxU9UGwGHXJ0MccQs6gLegQIGBAB&biw=1280&bih=735&dpr=1.25&jbr=sep:0")
time.sleep(15) #login duration

#scroll down for load more data
for i in range(20):
    driver.execute_script("window.scrollTo(0,document.body.scrollHeight)")
    time.sleep(0.5)

#titles 
titles = driver.find_elements(By.CSS_SELECTOR, "div.tNxQIb.PUpOsf")
job_titles = [t.text for t in titles]

job_card = driver.find_elements(By.CSS_SELECTOR, "div.EimVGf")
# Sab lists ki jagah ye use kar
jobs_list = []

for i, j in enumerate(job_card):
    try:
        # j.click() ki jagah ye use kar
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", j)
        time.sleep(0.5)
        j.click()

        # Show full description
        try:
            show_more = driver.find_element(By.CSS_SELECTOR, "div.TOQyFc.MmMIvd.jRKCUd")
            driver.execute_script("arguments[0].click();", show_more)
        except:
            pass

        job = {}  #job dictionary

        job["title"] = job_titles[i] if i < len(job_titles) else " "

        try:
            job["desc"] = driver.find_element(By.CSS_SELECTOR, "div.XFOJCe").text
        except:
            job["desc"] = " "

        details = driver.find_elements(By.CSS_SELECTOR, "div.aW97bd.wHYlTd span")
        company = " "
        location = " "
        for d in details:
            try:
                text = driver.execute_script("return arguments[0].innerText;", d)  # ✅
                if not text or text.strip().startswith("via"):
                    continue
                if re.search(r'Gujarat|Ahmedabad|Gandhinagar|Bhavnagar|Valsad|Vadodara|Nadiad|Rajkot|Jamnagar|Kheda|Nashik|Vapi|Shanti Nagar|Thangadh|Surat|Udaipur|Bharuch|Indore|Navsri|Mount Abu|Jodhpur', text, re.IGNORECASE):
                    location = text.strip()
                else:
                    company = text.strip()
            except:
                continue
        job["company"] = company
        job["location"] = location

        upload_time = " "
        salary = " "
        job_type = " "
        details1 = driver.find_elements(By.CSS_SELECTOR, "div.fpMzBb")
        for d in details1:
            try:
                text = driver.execute_script("return arguments[0].innerText;", d)
                if not text:
                    continue
                parts = text.strip().split('\n')
                for part in parts:
                    part = part.strip()
                    if re.search(r'\d+\s+(day|hour|week|month)', part, re.IGNORECASE):
                        upload_time = part
                    elif re.search(r'₹|LPA|lakh|k/month', part, re.IGNORECASE):
                        salary = part
                    elif re.search(r'full.time|part.time|internship|contract', part, re.IGNORECASE):
                        job_type = part
            except:
                continue
        job["upload_time"] = upload_time
        job["salary"] = salary
        job["job_type"] = job_type

        time.sleep(1)
        ratings_text = []
        rating_container = driver.find_elements(By.CSS_SELECTOR, "div.cXrwtf")
        for container in rating_container:

            rating_items = container.find_elements(By.CSS_SELECTOR, "a.SRK4ge")
            for r in rating_items:
                try:
                    source = r.find_element(By.CSS_SELECTOR, "div.HbJDWb").text
                    value = r.find_element(By.CSS_SELECTOR, "div.kgGzCe").text
                    ratings_text.append(f"{source}: {value}")
                except:
                    continue
        job["ratings"] = ", ".join(ratings_text) if ratings_text else "N/A"

        jobs_list.append(job)
        print(f"{i+1}. {job_titles[i]} ✅")

    except Exception as e:
        jobs_list.append({
            "title":" ",
            "desc": " ", "company": " ", "location": " ",
            "upload_time": " ", "salary": " ", "job_type": " ", "ratings": "N/A"
        })
        print(f"{i+1}. ❌ {e}")
        traceback.print_exc()
        continue

# save into JSON
#import json
#with open(r"C:\Users\JAY LODHA\web-scraping-project\data\raw\Data_Scientist_Jobs.json", "w", encoding="utf-8-sig") as f:
#    json.dump(jobs_list, f, ensure_ascii=False, indent=4)

print(f"Total jobs saved: {len(jobs_list)}")


In [2]:
import json
import pandas as pd

with open(r"C:\Users\JAY LODHA\web-scraping-project\data\raw\Data_Scientist_Jobs.json", "r", encoding="utf-8-sig") as f:
    data = json.load(f)

df = pd.DataFrame(data)
df.head()

,title,desc,company,location,upload_time,salary,job_type,ratings
0,Statistician / Data Scientist,Job description\nAs a Statistician / Data Scie...,Radix Analytics Pvt Ltd,"Ahmedabad, Gujarat",1 day ago,,Full–time,Indeed: 3.0/5
1,Data Science: AI/ML / Statistics / OR,Job description\nKey Responsibilities\n• Desig...,,,7 days ago,,Full–time,Indeed: 3.0/5
2,Predictive Analytics Data Scientist,Job description\nPredictive Analytics Data Sci...,Adani Enterprises Ltd,"Ahmedabad, Gujarat",7 days ago,,Full–time,"AmbitionBox: 3.6/5, Glassdoor: 3.5/5"
3,Senior Clinical Data Scientist,Job description\nAbout the Company\n\nData Sci...,CosMicIT,"Ahmedabad, Gujarat",,,,Glassdoor: 3.4/5
4,Data Science Developer,Job description\nSparks to Ideas IT Company is...,Sparks To Ideas Hiring For Sparks To Ideas,"Ahmedabad, Gujarat",,₹1L–₹2L a year,Full–time,N/A


In [7]:
df.replace([" ","N/A",""],None,inplace=True)
df['desc'].isnull().value_counts()

desc
False    135
True      28
Name: count, dtype: int64

In [5]:
import json
import pandas as pd

with open(r"C:\Users\JAY LODHA\web-scraping-project\data\raw\wol3d_bambu_products.json", "r", encoding="utf-8-sig") as f:
    data_json = json.load(f)

data = pd.DataFrame(data_json)
data.head()
data.to_csv(r"C:\Users\JAY LODHA\web-scraping-project\data\raw\wol3d_bambu_products.csv", index=False, encoding='utf-8-sig')